In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os 
import sys

current_dir = os.getcwd()

if current_dir.endswith('notebooks'):
    project_root = os.path.abspath(os.path.join(current_dir, '..'))
    os.chdir(project_root)
elif current_dir.endswith('thesis'):
    project_root = os.path.abspath(os.path.join(current_dir, 'Initial_project'))
    os.chdir(project_root)
else:
    project_root = current_dir

if project_root not in sys.path:
    sys.path.append(project_root)

print('Current working dir: ', os.getcwd())

In [ ]:
from src.data.dataloader import DataModule

config_path = 'configs/config.yml'
print(f'Loading the configuration data from {config_path}')
data_module = DataModule(config_path=config_path)
data_module.setup()
print(f'Found {data_module.classes} classes')

In [ ]:
train_loader = data_module.get_train_dataloader()
val_loader = data_module.get_val_dataloader()
print(f'Found {len(train_loader)} train batches')
print(f'Found {len(val_loader)} validation batches')
print('-' * 40)
images, labels = next(iter(train_loader))
print(f'Shape of images batch: {images.shape}')
print(f'Shape of labels batc: {labels.shape}')
print(f'First 5 labels: {labels[:5]}')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm
from src.models.vit_pretrained_classifier import PretrainedVit

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

model = PretrainedVit(num_classes=2, freeze_base=True)
model = model.to(device)



In [ ]:
EPOCHS = 5
LEARNING_RATE = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.model.heads.head.parameters(), lr=LEARNING_RATE)

history_epochs = []
history_train_loss = []
history_val_acc = []



In [ ]:
print('Start Training')
print('-'*40)


for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    training_loop = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]')

    for images, labels in training_loop:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        training_loop.set_postfix(loss=loss.item())

    avg_train_loss = running_loss / len(train_loader)

    model.eval()
    correct_preds = 0
    total_preds = 0

    val_loop = tqdm(val_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Val]', leave=False)
    with torch.no_grad():
        for images, labels in val_loop:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total_preds += labels.size(0)
            correct_preds += (predicted == labels).sum().item()

    val_accuracy = correct_preds / total_preds
    history_epochs.append(epoch + 1)
    history_train_loss.append(avg_train_loss)
    history_val_acc.append(val_accuracy)

    print(f'Epoch {epoch+1} finished | Train Loss: {avg_train_loss:.4f} | Validation acc: {val_accuracy:.4f}\n')
